# Quantum neural networks with and without qang

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_qml.ipynb)

This notebook uses the published library (`pip install qang`, module `qang.qml`). A weight-conserving QNN keeps its data in one Hamming-weight sector, so the **qg filter** can keep only the shots that stayed in that sector. Every result below is shown **with qang** (filtered readout) and **without qang** (raw readout), with their difference.

What to expect (RESEARCH_NOTES §75–§80): no quantum advantage (these models are classically simulable); trained on a simulator and run under T1 relaxation, qang recovers the noiseless accuracy exactly; trained under the calibrated noise, a model without qang catches up; dephasing and unequal T1 are not corrected by the filter.

## 1. Install

In [ ]:
!pip install -q -U "qang>=0.6.0" scikit-learn
import qang
print('qang', qang.__version__)

## 2. Data: breast cancer, 4 PCA features scaled to [-1, 1]

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from qang.qml import WeightQNN, compare_qang, kept_fraction

d = load_breast_cancer()
rng = np.random.default_rng(0)
idx = np.concatenate([rng.choice(np.where(d.target == c)[0], 100, replace=False) for c in (0, 1)])
Xtr, Xte, ytr, yte = train_test_split(d.data[idx], d.target[idx], test_size=0.3, stratify=d.target[idx], random_state=0)
sc = StandardScaler().fit(Xtr); pca = PCA(4, random_state=0).fit(sc.transform(Xtr))
Ztr, Zte = pca.transform(sc.transform(Xtr)), pca.transform(sc.transform(Xte))
lo, hi = Ztr.min(0), Ztr.max(0)
scale = lambda Z: np.clip(2 * (Z - lo) / (hi - lo) - 1, -1, 1)
Xtr, Xte = scale(Ztr), scale(Zte)
print(Xtr.shape, Xte.shape)

def show(rows, cols=('reading', 'with qang', 'without qang', 'difference')):
    print(f"{cols[0]:<34}{cols[1]:>11}{cols[2]:>13}{cols[3]:>12}")
    for name, q, r in rows:
        print(f"{name:<34}{q:>11.3f}{r:>13.3f}{q - r:>+12.3f}")

## 3. Weight-1 QNN, trained without noise, run under T1 (gamma = 0.08)

With qang the readout is exactly the noiseless one (F1); the price is the discarded shots, known in advance: (1 - gamma)^(weight x depth).

In [ ]:
GAMMA = 0.08
m1 = WeightQNN(n_qubits=5, weight=1).fit(Xtr, ytr, epochs=120, seed=0)
exact1 = m1.score(Xte, yte)
show([("weight 1, T1, trained clean", m1.score(Xte, yte, gamma=GAMMA, qang=True), m1.score(Xte, yte, gamma=GAMMA, qang=False))])
print(f"noiseless accuracy {exact1:.3f}; kept fraction with qang {kept_fraction(GAMMA, 1, m1.depth):.3f}")

## 4. Weight-2 QNN: the effect of qang grows with the weight

A weight-2 state decays twice as fast, so without qang the readout is more biased; with qang it is still exact.

In [ ]:
m2 = WeightQNN(n_qubits=5, weight=2).fit(Xtr, ytr, epochs=120, seed=0)
exact2 = m2.score(Xte, yte)
show([("weight 2, T1, trained clean", m2.score(Xte, yte, gamma=GAMMA, qang=True), m2.score(Xte, yte, gamma=GAMMA, qang=False))])
print(f"noiseless accuracy {exact2:.3f}; kept fraction with qang {kept_fraction(GAMMA, 2, m2.depth):.3f}")

## 5. Noise-aware training: training under the noise, with and without qang

If the device noise is known and the model is trained under it, the model without qang learns the decay and catches up (§79–§80). This takes a couple of minutes.

In [ ]:
r = compare_qang(WeightQNN(5, 2), Xtr, ytr, Xte, yte, gamma=GAMMA, epochs=120, seed=0, noise_aware=True)
show([("weight 2, T1, trained under noise", r["with qang"], r["without qang"])])

## 6. What the filter does not correct: unequal T1 across qubits, and dephasing

In [ ]:
unequal = [0.04, 0.06, 0.08, 0.10, 0.12]
show([
    ("weight 1, unequal T1", m1.score(Xte, yte, gamma=unequal, qang=True), m1.score(Xte, yte, gamma=unequal, qang=False)),
    ("weight 1, T1 + dephasing 0.03", m1.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=True), m1.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=False)),
    ("weight 2, unequal T1", m2.score(Xte, yte, gamma=unequal, qang=True), m2.score(Xte, yte, gamma=unequal, qang=False)),
    ("weight 2, T1 + dephasing 0.03", m2.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=True), m2.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=False)),
])
print(f"noiseless: weight 1 {exact1:.3f}, weight 2 {exact2:.3f}")

## 7. Finite shots (200 per input)

In [ ]:
show([
    ("weight 1, T1, 200 shots", m1.score(Xte, yte, gamma=GAMMA, qang=True, shots=200, seed=1), m1.score(Xte, yte, gamma=GAMMA, qang=False, shots=200, seed=1)),
    ("weight 2, T1, 200 shots", m2.score(Xte, yte, gamma=GAMMA, qang=True, shots=200, seed=1), m2.score(Xte, yte, gamma=GAMMA, qang=False, shots=200, seed=1)),
])

## Summary

* **Trained on a simulator, run under T1:** qang is decisive, exact under equal T1, and more so at higher weight.
* **Trained under the calibrated noise:** with and without qang are within about one test sample.
* **Cost:** a known fraction of the shots, 1 - (1 - gamma)^(weight x depth).
* **Limits:** unequal T1 and dephasing are not corrected; no quantum advantage.

Details, pre-registered predictions and failures: `RESEARCH_NOTES.md` §75–§80, scripts in `examples/qnn_*_qg.py`.